# 02 - Extra checks used in the paper

This notebook reproduces the checks that are not in notebook 01:

1. Reproduction of the held-out ROC-AUC values.
2. Label-permutation test for the No-ocular feature set.
3. Ocular pattern table and the AUC ceiling of three binary symptoms.
4. Paper figures: ROC, SHAP stability, calibration.
5. Split sensitivity (20 other random splits).
6. Add-one-feature test.
7. Re-run of the screen-time models of the earlier association study.
8. Figure: stable SHAP vs. AUC gain.

The full run takes about 15-30 minutes on a laptop CPU (the permutation test is the slow part). `QUICK_TEST=1` runs a tiny version to check that the code works. Its numbers are not the paper numbers.

XGBoost results can move by about 0.001-0.002 between library versions (see `requirements.txt`).

In [ ]:
# Settings. Set QUICK_TEST=1 for a fast smoke test with few permutations (numbers will NOT match the paper).
import os
from pathlib import Path
CSV = Path(os.environ.get("DRY_EYE_CSV", "../data/Dry_Eye_Dataset.csv"))   # raw Kaggle CSV
OUT = Path(os.environ.get("OUT_DIR", "../outputs/extra"))                  # tables and json
FIG = Path(os.environ.get("FIG_DIR", "../outputs/extra/figures"))          # figures
QUICK = os.environ.get("QUICK_TEST", "0") == "1"
OUT.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
print("CSV:", CSV.resolve(), "| exists:", CSV.exists(), "| QUICK_TEST:", QUICK)

## 0. Load and clean the data (same steps as notebook 01)

In [ ]:
import sys, json, time, os
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.base import clone
from xgboost import XGBClassifier

SEED = 42
csv, out = CSV, OUT
figdir = FIG
log = {}

# ---------------- preprocessing identical to the notebook ----------------
df = pd.read_csv(csv)
bp = df["Blood pressure"].astype(str).str.strip().str.extract(r"^(\d{2,3})/(\d{2,3})$").astype(int)
df["Systolic BP"], df["Diastolic BP"] = bp[0], bp[1]
df = df.drop(columns=["Blood pressure"])
yn = ["Sleep disorder","Wake up during night","Feel sleepy during day","Caffeine consumption",
      "Alcohol consumption","Smoking","Medical issue","Ongoing medication","Smart device before bed",
      "Blue-light filter","Discomfort Eye-strain","Redness in eye","Itchiness/Irritation in eye"]
for c in yn: df[c] = df[c].astype(str).str.strip().map({"N":0,"Y":1})
df["Gender_Male"] = (df["Gender"].astype(str).str.strip()=="M").astype(int); df = df.drop(columns=["Gender"])
df["Dry_Eye"] = df["Dry Eye Disease"].astype(str).str.strip().map({"N":0,"Y":1}); df = df.drop(columns=["Dry Eye Disease"])
df = df.loc[~(df["Systolic BP"] <= df["Diastolic BP"])].reset_index(drop=True)
log["n_rows"] = len(df)

G = {"Demographic":["Gender_Male","Age","Height","Weight"],
     "Sleep_Health":["Sleep duration","Sleep quality","Stress level","Systolic BP","Diastolic BP","Heart rate",
                     "Sleep disorder","Wake up during night","Feel sleepy during day","Medical issue","Ongoing medication"],
     "Lifestyle":["Daily steps","Physical activity","Caffeine consumption","Alcohol consumption","Smoking"],
     "Digital_Exposure":["Smart device before bed","Average screen time","Blue-light filter"],
     "Ocular_Symptoms":["Discomfort Eye-strain","Redness in eye","Itchiness/Irritation in eye"]}
ALL = sum(G.values(), []); OC = G["Ocular_Symptoms"]; NO = [f for f in ALL if f not in OC]
CFG = {"All Features":ALL, "No Ocular Symptoms":NO, "Ocular Symptoms Only":OC}

X = df.drop(columns=["Dry_Eye"]); y = df["Dry_Eye"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)

lr = Pipeline([("scaler",StandardScaler()),("classifier",LogisticRegression(max_iter=2000,class_weight="balanced",random_state=SEED))])
xgb = XGBClassifier(n_estimators=250,learning_rate=0.05,max_depth=4,min_child_weight=1,subsample=0.85,colsample_bytree=0.85,
                    objective="binary:logistic",eval_metric="logloss",tree_method="hist",random_state=SEED,n_jobs=-1,verbosity=0)
MODELS = {"Logistic Regression":lr, "XGBoost":xgb}
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)



## 1. Reproduce the held-out ROC-AUC values

In [ ]:
# ---------------- 1. reproduction of held-out ROC-AUC ----------------
NB = {("Logistic Regression","All Features"):0.6113,("Logistic Regression","No Ocular Symptoms"):0.5117,
      ("Logistic Regression","Ocular Symptoms Only"):0.6069,("XGBoost","All Features"):0.6016,
      ("XGBoost","No Ocular Symptoms"):0.5045,("XGBoost","Ocular Symptoms Only"):0.6038}
probs = {}; log["reproduction"] = {}
for m, est in MODELS.items():
    for c, feats in CFG.items():
        f = clone(est).fit(Xtr[feats], ytr); p = f.predict_proba(Xte[feats])[:,1]; probs[(m,c)] = p
        a = roc_auc_score(yte, p); log["reproduction"][f"{m} | {c}"] = {"reproduced":round(a,4),"notebook":NB[(m,c)]}
print(json.dumps(log["reproduction"], indent=1))



## 2. Label-permutation test (No-ocular set)

In [ ]:
# ---------------- 2. permutation test (No-Ocular, 5-fold CV on training) ----------------
rng = np.random.default_rng(SEED)
N_PERM = {"Logistic Regression":20, "XGBoost":10} if QUICK else {"Logistic Regression":1000, "XGBoost":200}
log["permutation"] = {}
for m, est in MODELS.items():
    real = cross_val_score(clone(est), Xtr[NO], ytr, cv=cv5, scoring="roc_auc").mean()
    null = []
    t = time.time()
    for i in range(N_PERM[m]):
        yp = pd.Series(rng.permutation(ytr.values), index=ytr.index)
        null.append(cross_val_score(clone(est), Xtr[NO], yp, cv=cv5, scoring="roc_auc").mean())
    null = np.array(null); p = (1 + np.sum(null >= real)) / (1 + len(null))
    log["permutation"][m] = {"n_perm":N_PERM[m],"real_auc":real,"null_mean":null.mean(),"null_sd":null.std(),
                             "null_q95":float(np.quantile(null,0.95)),"p_value":p,"seconds":time.time()-t}
    print(m, log["permutation"][m])



## 3. Ocular pattern table and AUC ceiling

In [ ]:
# ---------------- 3. ocular-symptom pattern table and AUC ceiling ----------------
pat = df[OC].astype(str).agg("".join, axis=1)
tab = df.groupby(pat)["Dry_Eye"].agg(["size","mean"]).rename(columns={"size":"n","mean":"ded_rate"})
tab["n_symptoms"] = [s.count("1") for s in tab.index]
tab = tab.sort_values(["n_symptoms","ded_rate"])
tab.to_csv(out/"ocular_pattern_table.csv")
print(tab)
# ceiling: score = pattern DED rate estimated on TRAIN, evaluated on TEST (no leakage)
ptr = Xtr[OC].astype(str).agg("".join, axis=1); pte = Xte[OC].astype(str).agg("".join, axis=1)
rate = ytr.groupby(ptr).mean()
ceiling_test = roc_auc_score(yte, pte.map(rate))
ceiling_insample = roc_auc_score(df["Dry_Eye"], pat.map(df.groupby(pat)["Dry_Eye"].mean()))
# number of symptoms as a score
nsym_test = roc_auc_score(yte, Xte[OC].sum(axis=1))
log["ceiling"] = {"pattern_rate_auc_test":ceiling_test,"pattern_rate_auc_full_insample":ceiling_insample,
                  "n_symptoms_auc_test":nsym_test,
                  "pattern_table":tab.reset_index().rename(columns={"index":"pattern"}).to_dict(orient="records")}
print(log["ceiling"])



## 4. Figures: ROC, SHAP stability, calibration

In [ ]:
# ---------------- 4a. ROC figure ----------------
plt.rcParams.update({"font.family":"serif","font.size":8,"axes.spines.top":False,"axes.spines.right":False,"pdf.fonttype":42})
COL = {"All Features":"#2a78d6","No Ocular Symptoms":"#eb6834","Ocular Symptoms Only":"#1baf7a"}
LAB = {"All Features":"All (26)","No Ocular Symptoms":"No ocular (23)","Ocular Symptoms Only":"Ocular only (3)"}
fig, ax = plt.subplots(1,2,figsize=(3.5,1.95),sharey=True)
for k,(m,title) in enumerate([("Logistic Regression","(a) LR"),("XGBoost","(b) XGBoost")]):
    for c in CFG:
        fpr,tpr,_ = roc_curve(yte, probs[(m,c)])
        ax[k].plot(fpr,tpr,lw=1.3,color=COL[c],label=LAB[c],
                   ls="--" if c=="Ocular Symptoms Only" else "-")
    ax[k].plot([0,1],[0,1],ls=":",lw=0.8,color="#777")
    ax[k].set_title(title,fontsize=7.5); ax[k].set_xlabel("False positive rate",fontsize=7)
    ax[k].legend(frameon=False,fontsize=5.6,loc="lower right",handlelength=1.6)
    ax[k].tick_params(labelsize=6)
ax[0].set_ylabel("True positive rate",fontsize=7)
fig.tight_layout(pad=0.3); fig.savefig(figdir/"roc_test.pdf",bbox_inches="tight"); fig.savefig(out/"roc_test.png",dpi=150,bbox_inches="tight")


plt.show()

In [ ]:
# ---------------- 4b. SHAP stability figure (15 validation-fold refits) ----------------
import shap
rs = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
Xall = Xtr[ALL]; rows = []
for s,(tri,vai) in enumerate(rs.split(Xall, ytr), start=1):
    fm = clone(xgb).fit(Xall.iloc[tri], ytr.iloc[tri])
    sv = shap.TreeExplainer(fm)(Xall.iloc[vai]).values
    imp = np.abs(sv).mean(axis=0)
    for f,v in zip(ALL, imp): rows.append({"Split":s,"Feature":f,"MeanAbsSHAP":v})
st = pd.DataFrame(rows); st.to_csv(out/"shap_stability_folds.csv",index=False)
order = st.groupby("Feature")["MeanAbsSHAP"].mean().sort_values(ascending=False).index[:10].tolist()
short = {"Discomfort Eye-strain":"Eye strain","Redness in eye":"Redness","Itchiness/Irritation in eye":"Itchiness",
         "Average screen time":"Screen time"}
fig, ax = plt.subplots(figsize=(3.5,1.9))
data = [st.loc[st.Feature==f,"MeanAbsSHAP"].values for f in order]
bp_ = ax.boxplot(data, widths=0.55, patch_artist=True, medianprops=dict(color="black",lw=1), flierprops=dict(markersize=2.5))
for i,b in enumerate(bp_["boxes"]): b.set_facecolor("#2a78d6" if order[i] in OC else "#b7d3f6"); b.set_linewidth(0.6)
ax.set_xticks(range(1,11),[short.get(f,f) for f in order],rotation=40,ha="right",fontsize=6)
ax.set_ylabel("Mean |SHAP|",fontsize=7); ax.tick_params(axis="y",labelsize=6)
fig.tight_layout(pad=0.3); fig.savefig(figdir/"shap_stability.pdf",bbox_inches="tight"); fig.savefig(out/"shap_stability.png",dpi=150,bbox_inches="tight")
log["shap_stability_mean"] = st.groupby("Feature")["MeanAbsSHAP"].mean().sort_values(ascending=False).round(4).to_dict()


plt.show()

In [ ]:
# ---------------- 4c. calibration figure (out-of-fold, training set) ----------------
fig, ax = plt.subplots(figsize=(3.0,2.1))
ax.plot([0,1],[0,1],ls=":",lw=0.8,color="#777",label="Perfect")
log["calibration_oof_mean"] = {}
for m,col,mk in [("Logistic Regression","#2a78d6","o"),("XGBoost","#eb6834","s")]:
    p = cross_val_predict(clone(MODELS[m]), Xtr[ALL], ytr, cv=cv5, method="predict_proba")[:,1]
    fo, mp = calibration_curve(ytr, p, n_bins=10, strategy="quantile")
    ax.plot(mp, fo, marker=mk, ms=3.5, lw=1.3, color=col, label=m)
    log["calibration_oof_mean"][m] = float(p.mean())
ax.set_xlabel("Mean predicted probability",fontsize=7); ax.set_ylabel("Observed DED rate",fontsize=7)
ax.set_xlim(0.2,0.85); ax.set_ylim(0.2,0.85); ax.tick_params(labelsize=6); ax.legend(frameon=False,fontsize=6)
fig.tight_layout(pad=0.3); fig.savefig(figdir/"calibration.pdf",bbox_inches="tight"); fig.savefig(out/"calibration.png",dpi=150,bbox_inches="tight")


plt.show()

In [ ]:
json.dump(log, open(out/"extra_checks.json","w"), indent=1, default=float)
print("saved", out/"extra_checks.json")

## 5. Split sensitivity

In [ ]:
from sklearn.model_selection import train_test_split
# Split sensitivity: repeat the held-out comparison on 10 other random splits at 80/20 and at 70/30.
rows = []
for frac, name in [(0.20, "80/20"), (0.30, "70/30")]:
    for seed in range(2 if QUICK else 10):
        a, b, c, d_ = train_test_split(X, y, test_size=frac, random_state=seed, stratify=y)
        for mname, est in MODELS.items():
            r = {"split": name, "seed": seed, "model": mname}
            for cfg, f in CFG.items():
                r[cfg] = roc_auc_score(d_, clone(est).fit(a[f], c).predict_proba(b[f])[:, 1])
            rows.append(r)
R = pd.DataFrame(rows)
R.to_csv(out / "split_sensitivity.csv", index=False)
R["drop"] = R["All Features"] - R["No Ocular Symptoms"]
R["gap"] = R["All Features"] - R["Ocular Symptoms Only"]
display(R.groupby(["split", "model"])[["All Features", "No Ocular Symptoms", "Ocular Symptoms Only", "drop", "gap"]].agg(["mean", "std"]).round(4).T)
print("Loss without ocular symptoms: min %.4f, max %.4f" % (R["drop"].min(), R["drop"].max()))
print("Mean All-vs-Ocular gap per group: min %.4f, max %.4f" % (R.groupby(["split", "model"])["gap"].mean().min(), R.groupby(["split", "model"])["gap"].mean().max()))

## 6. Add-one-feature test

In [ ]:
# Add-one-feature test: ocular-only baseline vs ocular + ONE non-ocular feature (LR).
# 5x5 repeated CV on the training set (25 paired splits), Nadeau-Bengio corrected resampled t-test, Holm correction.
from scipy.stats import t as tdist
mk = lambda: Pipeline([("s", StandardScaler()), ("c", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))])
cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
splits = list(cv_rep.split(Xtr, ytr))
def cvauc(cols):
    return np.array([roc_auc_score(ytr.iloc[v], mk().fit(Xtr.iloc[t][cols], ytr.iloc[t]).predict_proba(Xtr.iloc[v][cols])[:, 1]) for t, v in splits])
base = cvauc(OC); rows = []
for f in (NO[:3] if QUICK else NO):
    a_ = cvauc(OC + [f]); dd = a_ - base
    k = len(dd); var = dd.var(ddof=1) * (1 / k + 0.25)          # n_test / n_train = 1/4
    pv = 2 * tdist.sf(abs(dd.mean() / np.sqrt(var)), k - 1)
    # ci_lo / ci_hi = middle 95% of the 25 split-level differences (NOT a confidence interval of the mean)
    rows.append(dict(feature=f, mean_delta=dd.mean(), ci_lo=np.percentile(dd, 2.5), ci_hi=np.percentile(dd, 97.5), p=pv))
r_add = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
mm = len(r_add); r_add["p_holm"] = np.minimum(1, np.maximum.accumulate((mm - np.arange(mm)) * r_add.p.values))
r_add.to_csv(out / "add_one_feature.csv", index=False)
display(r_add.round(4).head(8))
test_auc = {}
for name, cols in [("ocular", OC), ("ocular+screen", OC + ["Average screen time"]), ("ocular+all non-ocular", OC + NO)]:
    test_auc[name] = round(roc_auc_score(yte, mk().fit(Xtr[cols], ytr).predict_proba(Xte[cols])[:, 1]), 4)
print("baseline CV AUC:", round(base.mean(), 4)); print("held-out AUC:", test_auc)
json.dump({"baseline_cv_auc": float(base.mean()), "held_out_auc": test_auc}, open(out / "add_one_feature_test_auc.json", "w"), indent=1)

## 7. Re-run of the screen-time association models (needs the raw CSV)

In [ ]:
import sys, json, numpy as np, pandas as pd
# Re-run of the screen-time models of the earlier association study, on the raw CSV (20,000 rows).
# This cell uses its own variables (d, X, y); run it after the cells above.
import statsmodels.api as sm
from statsmodels.formula.api import logit
from scipy import stats
d=pd.read_csv(CSV)
d.columns=[c.strip() for c in d.columns]
y=(d['Dry Eye Disease']=='Y').astype(int)
bp=d['Blood pressure'].str.split('/',expand=True).astype(float)
X=pd.DataFrame({'y':y,'screen':d['Average screen time'],'age':d['Age'],
 'male':(d['Gender']=='M').astype(int),'sleep':d['Sleep duration'],'sbp':bp[0],'dbp':bp[1],
 'bmi':d['Weight']/(d['Height']/100)**2})
for k,c in {'sd':'Sleep disorder','wake':'Wake up during night','sleepy':'Feel sleepy during day','caf':'Caffeine consumption','alc':'Alcohol consumption','smk':'Smoking','med':'Medical issue','meds':'Ongoing medication','dev':'Smart device before bed','strain':'Discomfort Eye-strain','red':'Redness in eye','itch':'Itchiness/Irritation in eye'}.items():
    X[k]=(d[c]=='Y').astype(int)
st_out={}
st_out['n']=len(X); st_out['ded']=int(y.sum()); st_out['ded_pct']=round(100*y.mean(),1)
st_out['female_ded']=int(((d.Gender=='F')&(y==1)).sum()); st_out['strain_yes_ded']=int(((X.strain==1)&(y==1)).sum())
st_out['screen_mean']=[round(X.screen.mean(),2),round(X.screen.std(),2),X.screen.min(),X.screen.max()]
noc='age male sleep sbp dbp bmi sd wake sleepy caf alc smk med meds dev'.split()
oc=['strain','red','itch']
def fit(f,extra=''):
    m=logit('y ~ '+f,X).fit(disp=0); return m
def row(m,v='screen'):
    b=m.params[v]; lo,hi=m.conf_int().loc[v]; return [round(float(np.exp(b)),3),round(float(np.exp(lo)),3),round(float(np.exp(hi)),3),float(m.pvalues[v])]
st_out['M1']=row(fit('screen'))
st_out['M2']=row(fit('screen+age+male'))
st_out['M3_with_ocular']=row(fit('screen+'+'+'.join(noc+oc)))
st_out['M3_no_ocular_terms']=row(fit('screen+'+'+'.join(noc)))
# ocular only effect
m=fit('screen+'+'+'.join(noc+oc)); st_out['ocular_ORs']={v:round(float(np.exp(m.params[v])),2) for v in oc}
# tertiles
X['tert']=pd.qcut(X.screen,3,labels=['low','mid','high'])
mt=logit('y ~ C(tert, Treatment("low")) + age+male',X).fit(disp=0)
st_out['tertile_M2']={k:[round(float(np.exp(mt.params[k])),3),float(mt.pvalues[k])] for k in mt.params.index if 'tert' in k}
# within ocular strata: any symptom vs none
X['anyoc']=(X[oc].sum(axis=1)>0).astype(int)
for g,s in X.groupby('anyoc'):
    r=row(logit('y ~ screen',s).fit(disp=0)); st_out[f'screen_within_anyoc={g}']=[len(s)]+r
# screen vs ocular symptoms (is screen time associated with the symptoms?)
for v in oc:
    r=row(logit(f'{v} ~ screen',X).fit(disp=0)); st_out[f'screen_vs_{v}']=r
# sex interaction
mi=logit('y ~ screen*male + age',X).fit(disp=0); st_out['sex_interaction_p']=float(mi.pvalues['screen:male'])
st_out['screen_by_sex']={g:row(logit('y ~ screen+age',s).fit(disp=0)) for g,s in X.groupby('male')}
# nonlinearity: spline (natural cubic, df=4) LR test, M3 adj
f3='+'.join(noc+oc)
m0=logit('y ~ screen+'+f3,X).fit(disp=0); m1=logit('y ~ cr(screen,df=4)+'+f3,X).fit(disp=0)
lr=2*(m1.llf-m0.llf); st_out['spline_LR_p_adj']=float(stats.chi2.sf(lr,m1.df_model-m0.df_model))
ms0=logit('y ~ screen',X).fit(disp=0); ms1=logit('y ~ cr(screen,df=4)',X).fit(disp=0)
st_out['spline_LR_p_unadj']=float(stats.chi2.sf(2*(ms1.llf-ms0.llf),ms1.df_model-ms0.df_model))
# AUC of screen alone & added value
from sklearn.metrics import roc_auc_score
st_out['auc_screen_alone']=round(roc_auc_score(y,X.screen),4)
# permutation null for the OR p-value: shuffle screen 2000 times
rng=np.random.default_rng(42); Xs=X.copy(); ps=[]
obs=m.params.get('screen')
b0=logit('y ~ screen+'+f3,X).fit(disp=0).params['screen']
cnt=0;N=20 if QUICK else 500
for i in range(N):
    Xs['screen']=rng.permutation(X.screen.values)
    bb=logit('y ~ screen+'+f3,Xs).fit(disp=0).params['screen']
    cnt+= abs(bb)>=abs(b0)
st_out['perm_p_M3']=(cnt+1)/(N+1)
# binary-column check of balance: DED rate by screen decile
X['dec']=pd.qcut(X.screen,10,duplicates='drop')
st_out['ded_rate_by_decile']=[round(v,3) for v in X.groupby('dec',observed=True).y.mean()]
json.dump(st_out,open(OUT/'screen_time_check.json','w'),indent=1,default=str)
print(json.dumps(st_out,indent=1,default=str))


## 8. Figure: stable SHAP vs. AUC gain

In [ ]:
# Figure for the paper: stable SHAP importance vs. AUC gain of each non-ocular feature.
from scipy.stats import spearmanr
shap_df = pd.read_csv(out / "shap_stability_folds.csv")
gain = pd.read_csv(out / "add_one_feature.csv").set_index("feature")
mm_ = shap_df.groupby("Feature")["MeanAbsSHAP"].agg(["mean", "std"])
dg = gain.join(mm_, how="inner")
rho, pv = spearmanr(dg["mean"], dg["mean_delta"])
summary = {"n_features": int(len(dg)), "spearman_rho": float(rho), "spearman_p": float(pv),
           "gain_min": float(dg.mean_delta.min()), "gain_max": float(dg.mean_delta.max()),
           "shap_min": float(dg["mean"].min()), "shap_max": float(dg["mean"].max()), "min_p_holm": float(dg.p_holm.min())}
json.dump(summary, open(out / "shap_vs_gain_summary.json", "w"), indent=1); print(summary)
plt.rcParams.update({"font.family": "serif", "font.size": 8, "pdf.fonttype": 42, "axes.spines.top": False, "axes.spines.right": False})
fig, ax = plt.subplots(figsize=(3.5, 2.3)); ax.axhline(0, color="#555", lw=0.8)
ax.errorbar(dg["mean"], dg["mean_delta"], xerr=dg["std"], fmt="o", ms=3.6, color="#1f4e8c", ecolor="#9db4d3", elinewidth=0.7, capsize=0, mew=0)
ax.set_ylim(-0.008, 0.008)
ax.set_xlabel("Mean |SHAP| (15 refits, bar = SD)", fontsize=7); ax.set_ylabel("$\\Delta$ROC-AUC when added to\nthe three symptoms", fontsize=7)
ax.tick_params(labelsize=6); ax.grid(axis="y", color="#e5e5e5", lw=0.5); ax.set_axisbelow(True)
fig.tight_layout(pad=0.3); fig.savefig(figdir / "shap_vs_gain.pdf", bbox_inches="tight"); plt.show()